## Workshop Lab 5b: Capstone Challenge - New Mexico Wildlife Proximity Analysis **SOLUTIONS**
**The mission:** Put your new coding knowledge to the test! For this notebook, you will:
- Fetch iNaturalist observations on **Mule Deer** in New Mexico
- Clean and export that data as a csv
- Convert the CSV into a point feature class
- Use Arcade expressions to visualize deer based on proximity to Black Bear sightings.

**How to approach this lab:**
You don't need to write all code from scratch! Professional developers rarely write code from memory. Instead they find working examples, copy them, and adapt the variables.
- **Reuse existing code:** Look back at **Notebook 3**, **Notebook 4**, **Notebook 5**, and your `.py` script files.
- **Follow the references:** Each step below tells you exactly which notebook cell or script file to open and copy from.
- **Adapt the parameters:** Update species names, place IDs, and file paths to match New Mexico Mule Deer.

*If you get stuck, check `05b_Solutions.ipynb` in the `02_notebooks` folder.*

### Step 1: Collect and Clean the Data
Query the iNaturalist API for **Mule Deer** in **New Mexico**, clean the observations, and save them as a CSV.

**Where to copy your code from:**
Open Notebook 5 (05_SharingAndAPIs.ipynb), look at the black bear API fetch. Copy the requests.get() structure, the parsing loop, and the DataFrame creation. Stop before the arcpy conversion part.

**What to adapt/check:**
- **`taxon_name`**: Change `"Ursus americanus"` to `"Odocoileus hemionus"` (Mule Deer).
- **`place_id`**: Use `9` (New Mexico).
- **`per_page`**: Set to `200`.
- **Output File Path**: Save your cleaned CSV to `"../04_data/clean/mule_deer_newmexico.csv"`.

In [ ]:
# Paste and modify your code here:

import requests
import pandas as pd
import os

# Define API endpoint and request parameters
url = "https://api.inaturalist.org/v1/observations"
params = {
    "taxon_name": "Odocoileus hemionus",
    "place_id": 9,
    "per_page": 200,
    "verifiable": "true"
}

# Make the API request and parse JSON response into a Pandas DataFrame
response = requests.get(url, params=params)
data = response.json()
observations = data.get("results", [])

records = []
for obs in observations:
    geo = obs.get("geojson")
    if geo and "coordinates" in geo:
        lon, lat = geo["coordinates"]
        records.append({
            "id": obs.get("id"),
            "observed_on": obs.get("observed_on"),
            "scientific_name": obs.get("species_guess"),
            "common_name": obs.get("common_name"),
            "latitude": lat,
            "longitude": lon
        })

df = pd.DataFrame(records)
df_clean = df.dropna(subset=["latitude", "longitude"]).copy()

os.makedirs("../04_data/clean", exist_ok=True)
output_csv = "../04_data/clean/mule_deer_newmexico.csv"
df_clean.to_csv(output_csv, index=False)
print(f"Success! Saved {len(df_clean)} records to {output_csv}")

### Step 2: Convert CSV into a Feature Class using ArcPy

Push your exported Mule Deer CSV into your `IntroToPythonAndArcade.gdb` as a Point Feature Class using `arcpy.management.XYTableToPoint`.

**Where to copy your code from:**
1. Open Notebook 3 (`03_data_wrangling.ipynb`), look at the top of the Cleaning Script chunk. Look at how input_path and output_path are defined as direct string variables (instead of using tool parameters like GetParameterAsText). Copy this.
2. Open your standalone Python script **`05_scripts/XYtoPoint.py`** (first used in **Notebook 3**, under the *Second Script Tool: XYtoPoint.py* section).
3. Copy the `arcpy.management.XYTableToPoint()` function and spatial reference setup.

**What to adapt:**
- **Input CSV (`in_table`)**: Point to `"../04_data/clean/mule_deer_newmexico.csv"`.
- **Output Feature Class (`out_feature_class`)**: Point directly to your project GDB: `"IntroToPythonAndArcade.gdb/Mule_Deer_Points"`.
- **Script Parameters vs. Direct Variables**: The script tool used `arcpy.GetParameterAsText()`. For this notebook cell, replace those with explicit string variables for `input_csv`, `gdb_path`, and `output_fc`.

In [ ]:
# Paste and modify your code here:

import arcpy
import os

# Define your input CSV path (from Step 1) and your output geodatabase feature class path
input_csv = os.path.abspath(os.path.join(os.getcwd(), "..", "04_data", "clean", "mule_deer_newmexico.csv"))
gdb_path = os.path.abspath(os.path.join(os.getcwd(), "..", "03_pro_project", "IntroToPythonAndArcade.gdb"))
output_fc = os.path.join(gdb_path, "Mule_Deer_Points")

# Define coordinate field names and spatial reference (WGS 1984 / EPSG 4326)
x_field = "longitude"
y_field = "latitude"
spatial_ref = arcpy.SpatialReference(4326)

if arcpy.Exists(output_fc):
    arcpy.management.Delete(output_fc)

# Execute arcpy.management.XYTableToPoint to create your feature class
arcpy.management.XYTableToPoint(
    in_table=input_csv,
    out_feature_class=output_fc,
    x_field=x_field,
    y_field=y_field,
    coordinate_system=spatial_ref
)

print(f"Successfully created point feature class: {output_fc}")

### Step 3: Symbology & Arcade Proximity in ArcGIS Pro

Switch over to **ArcGIS Pro**, add `Mule_Deer_Points` and `smblack_bears_cleaned` to your map, and apply Arcade expressions.

**Where to copy your expressions from:**
1. Open **Notebook 4 (`04_Arcade.ipynb`)**.
2. **Distance / Size Scaling:** Look at the chunk where we Vary Symbology by Attribute / Age of Observation.
3. **Popup Spatial Intersection:** Look at `Roadrunners500m`, which uses `FeatureSetByName()`, `BufferGeodetic()`, and `Intersects()`.

**What to adapt in ArcGIS Pro:**
- **Layer Context**: In the popup expression, update `FeatureSetByName($map, "all_roadrunners_cleaned_XYtoPoint")` to target your bear layer: `"smblack_bears_cleaned"`.
- **Buffer Distance**: Adjust the `BufferGeodetic` distance (e.g., 1000 meters) to evaluate deer proximity to known bear sightings.

#### Step 3 Solutions

**Popup: black bears within 1000 m of each deer**

In ArcGIS Pro, right click `Mule_Deer_Points`, then `Configure Popups`, then Expressions, then New. Name it `Bears1000m` and paste:

```
// Get the black bear layer from the map
var bears = FeatureSetByName($map, "smblack_bears_cleaned");

// Create a 1000-meter geodesic buffer around the clicked deer point
var search_area = BufferGeodetic($feature, 1000, 'meters');

// Count the bear points that fall inside that buffer
var nearby = Intersects(bears, search_area);
var count_nearby = Count(nearby);

if (count_nearby > 0) {
    return count_nearby + " Black Bear(s) recorded within 1000m of this deer.";
} else {
    return "No black bears recorded within 1000m.";
}
```

Then add a Text block to the popup and insert the expression as a field, the same as in Notebook 4. The layer name in `FeatureSetByName` must match the bear layer's name in your Contents pane exactly.

**Size symbology from age of observation**

The deer file has no `obs_year` column, so the year has to come from `observed_on` (a text date like `2019-10-11`). In the symbology pane, use Vary Symbology by Attribute, then Size, then the expression editor, and paste:

```
// Skip records with no date
if (IsEmpty($feature.observed_on)) { return null; }

// Turn the text into a date, take its year, and subtract this year
return Year(Date($feature.observed_on)) - Year(Today());
```

Then check "Enable size range" and "Show legend".